In [2]:
# Z-Score Based Cryptocurrency Investment Strategy Dashboard

# Hide code by default
from IPython.display import HTML, display
display(HTML('''
<script>
// Hide all code cells by default
$(document).ready(function() {
    $('div.input').hide();
});
</script>
'''))

# Import necessary packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
import os
import ipywidgets as widgets
from IPython.display import display, clear_output

# Import our modules
import settings
from zscore_strategy import ZScoreInvestmentStrategy
from data_fetcher import get_weekly_data, fetch_cmc_historical_data, generate_sample_data, save_to_csv

# Create interactive widgets for parameters
crypto_dropdown = widgets.Dropdown(
    options=list(settings.AVAILABLE_CRYPTOCURRENCIES.keys()),
    value=settings.CRYPTO_SYMBOL,
    description='Cryptocurrency:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='300px')
)

window_slider = widgets.IntSlider(
    value=settings.WINDOW_SIZE,
    min=4,
    max=52,
    step=1,
    description='Window Size (weeks):',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)

investment_input = widgets.FloatText(
    value=settings.WEEKLY_INVESTMENT,
    description='Weekly Investment ($):',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='300px')
)

data_source_toggle = widgets.RadioButtons(
    options=['API Data', 'Sample Data'],
    value='API Data',
    description='Data Source:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='300px'),
    disabled=False
)

run_button = widgets.Button(
    description='Run Strategy',
    button_style='success',
    tooltip='Click to run the strategy with the selected parameters',
    icon='play'
)

show_code_button = widgets.Button(
    description='Show/Hide Code',
    button_style='info',
    tooltip='Toggle display of code cells',
    icon='code'
)

# Add a header and description
header = widgets.HTML(
    value="<h1>Z-Score Cryptocurrency Investment Strategy</h1>"
    "<p>This dashboard allows you to test a dollar-cost averaging (DCA) strategy based on z-score statistics. "
    "The strategy allocates more investment when prices are statistically undervalued.</p>"
)

# Container for output
output_area = widgets.Output()
status_area = widgets.Output()

# Variable to store results
results_df = None
strategy_instance = None
crypto_data = None

# Function to toggle code visibility
def toggle_code(b):
    display(HTML('''
    <script>
    var code_shown = false;
    $(document).ready(function() {
        $('div.input').each(function(){
            if (code_shown) {
                $(this).hide();
            } else {
                $(this).show();
            }
        });
        code_shown = !code_shown;
    });
    </script>
    '''))

# Function to run the strategy
def run_strategy(b):
    global results_df, strategy_instance, crypto_data
    
    with status_area:
        clear_output()
        print(f"Running strategy...")
        print(f"- Cryptocurrency: {crypto_dropdown.value}")
        print(f"- Window Size: {window_slider.value} weeks")
        print(f"- Weekly Investment: ${investment_input.value}")
        print(f"- Data Source: {data_source_toggle.value}")
    
    with output_area:
        clear_output()
        
        # Update settings
        crypto_symbol = crypto_dropdown.value
        crypto_id = settings.AVAILABLE_CRYPTOCURRENCIES[crypto_symbol]['id']
        crypto_name = settings.AVAILABLE_CRYPTOCURRENCIES[crypto_symbol]['name']
        
        # Get data based on selection
        if data_source_toggle.value == 'API Data':
            # Try to load data from API
            print(f"Fetching historical data for {crypto_name} (ID: {crypto_id})...")
            df = fetch_cmc_historical_data(crypto_id)
            if df is not None:
                crypto_data = df
                # Save to CSV
                save_to_csv(df, crypto_symbol)
            else:
                print("Failed to get data from API. Using sample data instead.")
                crypto_data = generate_sample_data()
        else:
            # Generate sample data
            print("Generating sample data...")
            crypto_data = generate_sample_data()
        
        # Create strategy instance with selected parameters
        strategy_instance = ZScoreInvestmentStrategy(
            window_size=window_slider.value,
            weekly_investment=investment_input.value
        )
        
        # Preview the data
        print("\nData Preview:")
        print("=============")
        display(crypto_data.head())
        print(f"Total records: {len(crypto_data)}")
        
        # Run backtest
        print("\nRunning backtest...")
        results_df = strategy_instance.backtest(crypto_data)
        
        # Plot results
        plt.figure(figsize=(10, 14))
        strategy_instance.plot_results(results_df, crypto_symbol)
        
        # Calculate performance metrics
        initial_investment = strategy_instance.weekly_investment * len(results_df)
        final_portfolio = results_df['portfolio_value'].iloc[-1]
        roi = (final_portfolio / initial_investment - 1) * 100
        
        print("\nStrategy Performance:")
        print("=====================")
        print(f"Initial Investment: ${initial_investment}")
        print(f"Final Portfolio Value: ${final_portfolio:.2f}")
        print(f"ROI: {roi:.2f}%")
        print(f"{crypto_symbol} Holdings: {results_df['crypto_holdings'].iloc[-1]:.6f}")
        print(f"Remaining Cash: ${results_df['cash_available'].iloc[-1]:.2f}")
        
        # Compare to simple DCA
        comparison = strategy_instance.compare_to_dca(crypto_data, results_df)
        
        print("\nComparison to Simple DCA Strategy:")
        print("================================")
        print(f"Simple DCA Investment: ${comparison['dca_investment']}")
        print(f"Simple DCA Final Value: ${comparison['dca_final_value']:.2f}")
        print(f"Simple DCA ROI: {comparison['dca_roi']:.2f}%")
        print(f"Z-Score Strategy Outperformance: {comparison['outperformance']:.2f}%")
        
        # Save results
        os.makedirs('results', exist_ok=True)
        today = datetime.now().strftime('%Y%m%d')
        results_file = f'results/{crypto_symbol}_zscore_strategy_{today}.csv'
        results_df.to_csv(results_file, index=False)
        print(f"\nResults saved to {results_file}")

# Add event handlers
show_code_button.on_click(toggle_code)
run_button.on_click(run_strategy)

# Create dashboard layout
parameter_box = widgets.VBox([
    widgets.HBox([crypto_dropdown, investment_input]),
    widgets.HBox([window_slider, data_source_toggle]),
    widgets.HBox([run_button, show_code_button])
])

# Display the dashboard
display(header)
display(parameter_box)
display(status_area)
display(output_area)

# Run the strategy with default parameters when the notebook opens
run_button.click()

ModuleNotFoundError: No module named 'ipywidgets'